# Notebook 03 - Model A vs Model B: comparison, bias analysis and interpretation
This notebook loads the saved results of notebooks 01 and 02 (no retraining of the main models). It then:
1. compares the six configurations (Chapter 4.4);
2. measures the gain from the OSM/GIS variables on the same test records and on paired CV folds;
3. runs the assessment-ratio (consistency and bias) analysis (Chapter 4.5);
4. interprets the models: coefficients, permutation importance, SHAP, partial dependence and a worked example for one
   property (Chapter 4.6);
5. runs supplementary checks (spatially grouped CV, a raw-target model, and results without the flagged records);
6. builds the Chapter 4 draft (`report/Chapter4_draft.docx`).

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from scipy import stats

from src import config as C, data as D, modelling as M, runner as R, plots as P, interpret as I
P.setup_style()
pd.set_option("display.max_columns", 40, "display.width", 200, "display.float_format", lambda v: f"{v:,.3f}")
T, F = C.TABLES_DIR, C.FIGURES_DIR
def show(fig, name):
    P.save(fig, name); plt.show()
from sklearn.inspection import permutation_importance

In [2]:
data = D.load_and_prepare(verbose=False)
lab, train, test = data["labelled"], data["train"], data["test"]
outA, outB = R.load_set("A"), R.load_set("B")
assert outA["test_ids"] == outB["test_ids"] == list(test["Property_ID"]), "A and B must share the test records"
sets, screened, _ = M.feature_sets(train)
categories = M.condition_categories(lab["Condition"])
yte, ytr = test[C.TARGET].values, train[C.TARGET]
OUT = {"A": outA, "B": outB}

## 1. Six-configuration comparison (Chapter 4.4)

In [3]:
rows = []
for s, out in OUT.items():
    for k, r in out["results"].items():
        rows.append({"Set": s, "key": k, "Model set": C.MODEL_SETS[s], "Algorithm": C.MODEL_NAMES[k],
                     "Predictors": len(r["features"]), "CV RMSE": r["cv"]["CV RMSE"], "CV RMSE SD": r["cv"]["CV RMSE SD"],
                     "CV MAE": r["cv"]["CV MAE"], "CV R2 pooled": r["cv"]["CV R2 pooled"],
                     "Test RMSE": r["test"]["RMSE"], "Test MAE": r["test"]["MAE"], "Test R2": r["test"]["R2"],
                     "Test MAPE (%)": 100 * np.mean(np.abs(r["pred_test"] - yte) / yte)})
comp = pd.DataFrame(rows)
best = comp.sort_values("CV RMSE").iloc[0]
comp["Selected (lowest CV RMSE)"] = np.where((comp["Set"] == best["Set"]) & (comp["key"] == best["key"]), "Yes", "")
comp.drop(columns=["Set", "key"]).to_csv(T / "AB_six_model_comparison.csv", index=False)
comp.drop(columns=["Set", "key"])

,Model set,Algorithm,Predictors,CV RMSE,CV RMSE SD,CV MAE,CV R2 pooled,Test RMSE,Test MAE,Test R2,Test MAPE (%),Selected (lowest CV RMSE)
0,Model A - property only,Multiple Linear Regression,4,"12,417.444","10,289.739","4,104.119",0.702,"6,626.339","3,626.702",0.923,23.379,
1,Model A - property only,Random Forest,4,"11,609.659","6,224.479","3,995.526",0.800,"6,905.540","3,406.758",0.917,23.217,
2,Model A - property only,XGBoost,4,"11,663.288","6,722.302","4,042.048",0.792,"6,171.024","3,143.016",0.933,20.654,
3,Model B - property + GIS,Multiple Linear Regression,7,"13,119.307","11,917.382","4,086.432",0.641,"6,345.987","3,451.609",0.930,22.645,
4,Model B - property + GIS,Random Forest,15,"11,468.491","6,327.478","3,808.183",0.803,"5,973.176","2,804.682",0.938,20.037,
5,Model B - property + GIS,XGBoost,15,"11,247.784","6,249.478","3,739.854",0.810,"5,767.699","2,932.160",0.942,20.284,Yes


In [4]:
show(P.model_comparison_bars(comp), "AB_model_comparison.png")
print(f"Best configuration (selected on cross-validation, not on the test set): {best['Model set']} - {best['Algorithm']}")

Best configuration (selected on cross-validation, not on the test set): Model B - property + GIS - XGBoost


The best configuration is chosen on **cross-validated RMSE in the training set**, not on the test set. The
test set therefore stays an unbiased check. With about 100 test properties, small differences between models (a few
hundred rupees of RMSE) are within sampling noise. The paired analysis below is the fairer test of the GIS effect.

## 2. Does adding OSM/GIS variables improve the estimates? (A vs B)

In [5]:
gain, fold_rows = [], []
for k in C.ALGORITHMS:
    a, b = outA["results"][k], outB["results"][k]
    fa, fb = a["folds"].set_index("split"), b["folds"].set_index("split")
    d_rmse = fb["RMSE"] - fa["RMSE"]
    # absolute errors on the same test records -> Wilcoxon signed-rank test
    ea, eb = np.abs(a["pred_test"] - yte), np.abs(b["pred_test"] - yte)
    w_p = stats.wilcoxon(eb, ea).pvalue
    t_p = stats.ttest_rel(fb["RMSE"], fa["RMSE"]).pvalue
    gain.append({"Algorithm": C.MODEL_NAMES[k],
                 "Test RMSE A": a["test"]["RMSE"], "Test RMSE B": b["test"]["RMSE"],
                 "Test RMSE change (%)": 100 * (b["test"]["RMSE"] / a["test"]["RMSE"] - 1),
                 "Test MAE A": a["test"]["MAE"], "Test MAE B": b["test"]["MAE"],
                 "Test MAE change (%)": 100 * (b["test"]["MAE"] / a["test"]["MAE"] - 1),
                 "Test R2 A": a["test"]["R2"], "Test R2 B": b["test"]["R2"],
                 "Test: properties with smaller error in B (%)": 100 * np.mean(eb < ea),
                 "Wilcoxon p (test abs. errors)": w_p,
                 "CV RMSE change, mean (Rs.)": d_rmse.mean(), "CV RMSE change, SD": d_rmse.std(),
                 "CV RMSE change (%)": 100 * (fb["RMSE"].mean() / fa["RMSE"].mean() - 1),
                 "CV folds where B better (%)": 100 * np.mean(d_rmse < 0),
                 "Paired t-test p (50 CV folds)": t_p})
    fold_rows.append(pd.DataFrame({"key": k, "split": d_rmse.index, "RMSE change (B - A)": d_rmse.values}))
gain = pd.DataFrame(gain)
gain.to_csv(T / "AB_gis_gain.csv", index=False)
fold_diff = pd.concat(fold_rows)
show(P.paired_cv_differences(fold_diff), "AB_paired_cv_differences.png")
gain.T

,0,1,2
Algorithm,Multiple Linear Regression,Random Forest,XGBoost
Test RMSE A,"6,626.339","6,905.540","6,171.024"
Test RMSE B,"6,345.987","5,973.176","5,767.699"
Test RMSE change (%),-4.231,-13.502,-6.536
Test MAE A,"3,626.702","3,406.758","3,143.016"
Test MAE B,"3,451.609","2,804.682","2,932.160"
Test MAE change (%),-4.828,-17.673,-6.709
Test R2 A,0.923,0.917,0.933
Test R2 B,0.930,0.938,0.942
Test: properties with smaller error in B (%),60.396,53.465,54.455


**How to read this.** A negative change means Model B (with GIS) has a lower error than Model A. Because the CV
folds are identical for A and B, the 50 paired fold differences show both the **size** and the **consistency** of
any GIS gain (methodology 3.9). The p-values (paired t-test on folds, Wilcoxon on the test-set absolute errors) are
indicative only, because the repeated CV folds overlap.

## 3. Consistency and bias: assessment-ratio study (Chapter 4.5)

In [6]:
rr = []
for s, out in OUT.items():
    for k, r in out["results"].items():
        for sample, st in [("Hold-out test", r["ratio"]), ("Training (out-of-fold CV)", r["ratio_oof"])]:
            rr.append({"Model set": C.MODEL_SETS[s], "Algorithm": C.MODEL_NAMES[k], "Sample": sample, **st,
                       "IAAO check": M.iaao_verdict(st)})
ratio_all = pd.DataFrame(rr)
ratio_all.to_csv(T / "AB_ratio_study.csv", index=False)
ratio_all[["Model set", "Algorithm", "Sample", "n", "Median ratio", "COD (%)", "PRD", "PRB", "Within +/-20% (%)", "IAAO check"]]

,Model set,Algorithm,Sample,n,Median ratio,COD (%),PRD,PRB,Within +/-20% (%),IAAO check
0,Model A - property only,Multiple Linear Regression,Hold-out test,101,1.010,23.127,1.177,-0.139,78.218,level OK; COD > 15; PRD regressive
1,Model A - property only,Multiple Linear Regression,Training (out-of-fold CV),401,1.001,14.376,1.052,-0.005,77.556,level OK; uniformity OK; PRD regressive
2,Model A - property only,Random Forest,Hold-out test,101,1.001,23.190,1.158,-0.123,83.168,level OK; COD > 15; PRD regressive
3,Model A - property only,Random Forest,Training (out-of-fold CV),401,1.003,12.907,1.050,-0.014,81.047,level OK; uniformity OK; PRD regressive
4,Model A - property only,XGBoost,Hold-out test,101,1.015,20.279,1.151,-0.122,86.139,level OK; COD > 15; PRD regressive
5,Model A - property only,XGBoost,Training (out-of-fold CV),401,0.994,12.943,1.058,-0.020,81.047,level OK; uniformity OK; PRD regressive
6,Model B - property + GIS,Multiple Linear Regression,Hold-out test,101,1.020,22.046,1.166,-0.134,74.257,level OK; COD > 15; PRD regressive
7,Model B - property + GIS,Multiple Linear Regression,Training (out-of-fold CV),401,1.014,13.717,1.044,-0.000,80.549,level OK; uniformity OK; PRD regressive
8,Model B - property + GIS,Random Forest,Hold-out test,101,1.005,19.933,1.140,-0.118,81.188,level OK; COD > 15; PRD regressive
9,Model B - property + GIS,Random Forest,Training (out-of-fold CV),401,0.993,11.682,1.045,-0.012,84.788,level OK; uniformity OK; PRD regressive


In [7]:
bands, band_rows = {}, []
styles = {("A", "mlr"): ("#9AA5B1", "--"), ("B", "mlr"): ("#5B6C8F", "-"), ("A", best["key"]): ("#E0B07A", "--"),
          ("B", best["key"]): ("#C2772B", "-")}
for (s, k), (col, ls) in styles.items():
    t = M.ratio_by_value_band(yte, OUT[s]["results"][k]["pred_test"])
    bands[(f"{C.MODEL_SETS[s].split(' - ')[0]} {C.MODEL_SHORT[k]}", col, ls)] = t
    band_rows.append(t.assign(**{"Model set": C.MODEL_SETS[s], "Algorithm": C.MODEL_NAMES[k]}))
pd.concat(band_rows).to_csv(T / "AB_ratio_by_value_band.csv", index=False)
show(P.ratio_by_band(bands), "AB_ratio_by_value_band.png")
pd.concat(band_rows)

,Band,n,AV_min,AV_max,Median_ratio,Model set,Algorithm
0,Q1,26,"1,152.000","8,640.000",1.011,Model A - property only,Multiple Linear Regression
1,Q2,25,"9,000.000","16,320.000",1.108,Model A - property only,Multiple Linear Regression
2,Q3,25,"17,280.000","30,720.000",1.000,Model A - property only,Multiple Linear Regression
3,Q4,25,"32,640.000","115,200.000",0.911,Model A - property only,Multiple Linear Regression
0,Q1,26,"1,152.000","8,640.000",1.023,Model B - property + GIS,Multiple Linear Regression
1,Q2,25,"9,000.000","16,320.000",1.066,Model B - property + GIS,Multiple Linear Regression
2,Q3,25,"17,280.000","30,720.000",0.999,Model B - property + GIS,Multiple Linear Regression
3,Q4,25,"32,640.000","115,200.000",0.905,Model B - property + GIS,Multiple Linear Regression
0,Q1,26,"1,152.000","8,640.000",1.020,Model A - property only,XGBoost
1,Q2,25,"9,000.000","16,320.000",1.015,Model A - property only,XGBoost


### Sensitivity: ratio statistics without the flagged extreme-rate records

In [8]:
keep = (test["Flag_Extreme_AV_per_sqft"] == 0).values
sens = []
for s, out in OUT.items():
    for k, r in out["results"].items():
        st = M.ratio_statistics(yte[keep], r["pred_test"][keep])
        sens.append({"Model set": C.MODEL_SETS[s], "Algorithm": C.MODEL_NAMES[k], **M.metrics(yte[keep], r["pred_test"][keep]),
                     **{x: st[x] for x in ["n", "Median ratio", "COD (%)", "PRD", "PRB"]}, "IAAO check": M.iaao_verdict(st)})
sens = pd.DataFrame(sens)
sens.to_csv(T / "AB_ratio_without_flagged.csv", index=False)
print(f"Test records excluded: {(~keep).sum()}")
sens

Test records excluded: 3


,Model set,Algorithm,MAE,RMSE,R2,n,Median ratio,COD (%),PRD,PRB,IAAO check
0,Model A - property only,Multiple Linear Regression,"2,941.263","4,623.403",0.962,98,1.010,12.337,1.060,-0.039,level OK; uniformity OK; PRD regressive
1,Model A - property only,Random Forest,"2,669.032","4,743.105",0.960,98,1.002,11.728,1.038,-0.022,level OK; uniformity OK; PRD regressive
2,Model A - property only,XGBoost,"2,448.358","3,955.267",0.972,98,1.016,9.801,1.038,-0.025,level OK; uniformity OK; PRD regressive
3,Model B - property + GIS,Multiple Linear Regression,"2,789.578","4,390.633",0.965,98,1.021,11.509,1.053,-0.035,level OK; uniformity OK; PRD regressive
4,Model B - property + GIS,Random Forest,"2,078.836","3,412.465",0.979,98,1.005,9.540,1.030,-0.021,level OK; uniformity OK; PRD regressive
5,Model B - property + GIS,XGBoost,"2,292.496","3,806.039",0.974,98,0.999,9.830,1.031,-0.015,level OK; uniformity OK; PRD regressive


**Reading the ratio study.** The median ratio tests the overall *level* (do the models reproduce the CMC
values on average?). COD tests *uniformity*: how far individual ratios are from the median. PRD and PRB test
*vertical equity*. A PRD above 1.03 or a negative PRB means low-value properties receive relatively higher estimates
than high-value ones (regressive). The models are trained on CMC values, so they reproduce any systematic pattern in
those values. These figures show agreement and consistency with CMC assessments; they do not show market-value
accuracy.

## 4. Interpretation (Chapter 4.6)

### 4.1 MLR coefficients - Model A vs Model B side by side

In [9]:
olsA = pd.read_csv(T / "A_mlr_ols.csv"); olsB = pd.read_csv(T / "B_mlr_ols.csv")
fitA = pd.read_csv(T / "A_mlr_ols_fit.csv"); fitB = pd.read_csv(T / "B_mlr_ols_fit.csv")
coef = olsA[["Term", "Coefficient (log scale)", "p-value", "Readable effect"]].merge(
    olsB[["Term", "Coefficient (log scale)", "p-value", "Readable effect"]], on="Term", how="outer", suffixes=(" A", " B"))
order = {t: i for i, t in enumerate(olsB["Term"])}
coef = coef.sort_values("Term", key=lambda s: s.map(order)).reset_index(drop=True)
coef.to_csv(T / "AB_mlr_coefficients.csv", index=False)
display(pd.concat([fitA.assign(Model="A"), fitB.assign(Model="B")]))
coef

,n,R2 (log scale),Adj. R2 (log scale),F p-value,AIC,Condition number,Model
0,401,0.932,0.931,0.000,-49.766,305.304,A
0,401,0.935,0.934,0.000,-62.192,"38,028.009",B


,Term,Coefficient (log scale) A,p-value A,Readable effect A,Coefficient (log scale) B,p-value B,Readable effect B
0,const,3.508,0.000,NaN,3.399,0.000,NaN
1,Age,-0.001,0.279,-1.25 % per +10 years,-0.000,0.687,-0.48 % per +10 years
2,Log_Total_Floor_Area,0.980,0.000,+9.79 % per +10 % floor area (elasticity 0.980),0.976,0.000,+9.75 % per +10 % floor area (elasticity 0.976)
3,Num_Floors,-0.102,0.000,-9.68 % per extra floor (same total area),-0.088,0.000,-8.41 % per extra floor (same total area)
4,Dist_AB_Road,NaN,NaN,NaN,0.000,0.003,+1.24 % per +100 m
5,Dist_Railway,NaN,NaN,NaN,-0.000,0.875,-0.12 % per +100 m
6,Dist_Hospital,NaN,NaN,NaN,-0.000,0.787,-0.30 % per +100 m
7,Condition_Poor,-0.376,0.002,-31.32 % vs Fair,-0.352,0.003,-29.70 % vs Fair
8,Condition_Fair-,-0.257,0.000,-22.70 % vs Fair,-0.225,0.000,-20.11 % vs Fair
9,Condition_Fair+,0.064,0.025,+6.65 % vs Fair,0.093,0.002,+9.80 % vs Fair


### 4.2 Permutation importance (increase in CV RMSE when a variable is shuffled)

In [10]:
imp_tables = []
for s in ["A", "B"]:
    for k in C.ALGORITHMS:
        ps = R.permutation_summary(OUT[s]["results"][k]["perm"])
        imp_tables.append(ps.assign(**{"Model set": C.MODEL_SETS[s], "Algorithm": C.MODEL_NAMES[k]}))
imp = pd.concat(imp_tables)
imp.to_csv(T / "AB_permutation_importance.csv", index=False)
for s in ["A", "B"]:
    k = best["key"]
    ps = imp[(imp["Model set"] == C.MODEL_SETS[s]) & (imp["Algorithm"] == C.MODEL_NAMES[k])]
    show(P.importance_bars(ps, f"Permutation importance - {C.MODEL_SETS[s]}, {C.MODEL_NAMES[k]}"),
         f"AB_permutation_importance_{s}_{k}.png")
imp[imp["Model set"] == C.MODEL_SETS["B"]].pivot_table(index="Label", columns="Algorithm", values="Relative importance (%)") \
   .sort_values(C.MODEL_NAMES[best["key"]], ascending=False)

Algorithm,Multiple Linear Regression,Random Forest,XGBoost
Label,,,
Total floor area (sq ft),NaN,94.788,93.995
Number of floors,0.597,3.155,2.707
Condition grade,3.250,0.211,0.886
Distance to bus stop (m),NaN,0.198,0.850
Distance to pharmacy (m),NaN,0.367,0.769
Distance to A/B road (m),0.000,0.369,0.338
Distance to supermarket (m),NaN,0.021,0.190
Distance to playground (m),NaN,0.048,0.115
Distance to hospital (m),0.000,0.100,0.103


In [11]:
grp = imp[imp["Model set"] == C.MODEL_SETS["B"]].copy()
grp["Group"] = np.where(grp["Feature"].isin(C.GIS_FEATURES), "GIS / location (all distances)", "Property characteristics")
gshare = grp.groupby(["Algorithm", "Group"])["Relative importance (%)"].sum().unstack()
gshare.to_csv(T / "AB_importance_by_group.csv")
gshare

Group,GIS / location (all distances),Property characteristics
Algorithm,,
Multiple Linear Regression,0.000,100.000
Random Forest,1.759,98.241
XGBoost,2.413,97.587


### 4.3 SHAP - best tree model, Model B

In [12]:
tree_key = best["key"] if best["key"] != "mlr" else comp[comp["key"] != "mlr"].sort_values("CV RMSE").iloc[0]["key"]
mB = OUT["B"]["results"][tree_key]["model"]
fB = sets[("B", tree_key)]
sv, base, Xe = I.shap_values(mB, train[fB], train[fB])
shap_imp = I.shap_importance(sv)
shap_imp.to_csv(T / "AB_shap_importance_B.csv", index=False)
fig = I.shap_summary_plot(sv, Xe, f"SHAP summary - Model B, {C.MODEL_NAMES[tree_key]} (training properties)")
show(fig, "AB_shap_summary_B.png")
shap_imp

,Feature,Label,Mean |SHAP| (log scale),Typical effect on Annual Value (%)
0,Total_Floor_Area,Total floor area (sq ft),0.675,96.347
1,Condition,Condition grade,0.044,4.453
2,Dist_AB_Road,Distance to A/B road (m),0.025,2.506
3,Age,Building age (years),0.022,2.230
4,Dist_Bus_Stop,Distance to bus stop (m),0.020,2.019
5,Num_Floors,Number of floors,0.015,1.518
6,Dist_Pharmacy,Distance to pharmacy (m),0.014,1.425
7,Dist_Railway,Distance to railway station (m),0.013,1.260
8,Dist_Hospital,Distance to hospital (m),0.012,1.204
9,Dist_CBD,Distance to CBD (m),0.008,0.831


Each dot is a property. Its horizontal position is that variable's contribution to the (log) estimate,
relative to the average. The colour is the variable's value (red = high). The table converts the mean absolute
contribution into a typical percentage effect on Annual Value.

### 4.4 Partial dependence (average effect of one variable, others held at their observed values)

In [13]:
ps_b = imp[(imp["Model set"] == C.MODEL_SETS["B"]) & (imp["Algorithm"] == C.MODEL_NAMES[tree_key])]
pdp_feats = [f for f in ps_b["Feature"] if f != "Condition"][:6]
show(I.pdp_figure(mB, train[fB], pdp_feats, f"Partial dependence - Model B, {C.MODEL_NAMES[tree_key]}"), "AB_pdp_B.png")
pdp_feats

['Total_Floor_Area',
 'Num_Floors',
 'Dist_Bus_Stop',
 'Dist_Pharmacy',
 'Dist_AB_Road',
 'Dist_Supermarket']

### 4.5 Worked example: explaining one property's assessment
The example is a typical test property: the one whose CMC Annual Value is closest to the test-set median. Two
explanations are shown.
* **MLR (Model B)** gives an itemised, rule-like explanation. It starts from a *reference property* (median floor area,
  one floor, median age, Fair condition, median distances). Each recorded characteristic then moves the estimate by a
  fixed percentage that is the same for every property, much like an adjustment on a valuation sheet.
* **The best tree model** gives a SHAP explanation for this property.

In [14]:
ex_idx = int(np.argmin(np.abs(yte - np.median(yte))))
ex = test.iloc[[ex_idx]]
raw_vals = {c: ex.iloc[0][c] for c in ["Age", "Condition", "Total_Floor_Area", "Log_Total_Floor_Area", "Num_Floors"] + C.GIS_FEATURES}
raw_vals["Log_Total_Floor_Area"] = f"log({ex.iloc[0]['Total_Floor_Area']:,.0f} sq ft)"
ex_info = ex[["Property_ID", "Assessment_No", "Age", "Condition", "Total_Floor_Area", "Num_Floors", C.TARGET] + C.GIS_FEATURES].T
ex_info.to_csv(T / "AB_worked_example_property.csv", header=["Value"])
display(ex_info)

mlrB = OUT["B"]["results"]["mlr"]["model"]; fM = sets[("B", "mlr")]
contrib_m, base_m, ref = I.mlr_itemised(mlrB, train, ex, fM)
ref_desc = (f"Reference property ({train['Total_Floor_Area'].median():,.0f} sq ft, {ref['Num_Floors'].iloc[0]:.0f} floor, "
            f"{ref['Age'].iloc[0]:.0f} yrs, {C.CONDITION_REFERENCE}, median distances)")
steps_m = I.waterfall_steps(contrib_m, base_m, raw_vals, base_label=ref_desc)
steps_m.to_csv(T / "AB_worked_example_mlr.csv", index=False)
ref.assign(Total_Floor_Area=train["Total_Floor_Area"].median()).to_csv(T / "AB_worked_example_reference.csv", index=False)
show(P.waterfall(steps_m, f"MLR (Model B) explanation - property {ex.iloc[0]['Property_ID']}\n"
                          f"CMC Annual Value Rs. {ex.iloc[0][C.TARGET]:,.0f}"), "AB_worked_example_mlr.png")

sv_t, base_t, _ = I.shap_values(mB, train[fB], ex[fB])
steps_t = I.waterfall_steps(sv_t.iloc[0], base_t, raw_vals, top=6)
steps_t.to_csv(T / "AB_worked_example_tree.csv", index=False)
show(P.waterfall(steps_t, f"{C.MODEL_NAMES[tree_key]} (Model B) SHAP explanation - property {ex.iloc[0]['Property_ID']}\n"
                          f"CMC Annual Value Rs. {ex.iloc[0][C.TARGET]:,.0f}"), "AB_worked_example_tree.png")
assert abs(steps_m["End (Rs.)"].iloc[-1] - mlrB.predict(ex[fM])[0]) < 1
assert abs(steps_t["End (Rs.)"].iloc[-1] - mB.predict(ex[fB])[0]) / mB.predict(ex[fB])[0] < 0.01
steps_m[["Label", "Effect (%)", "Change (Rs.)", "End (Rs.)"]]

,53
Property_ID,DEM_00283
Assessment_No,94/3
Age,25.000
Condition,Fair+
Total_Floor_Area,684.000
Num_Floors,3.000
Annual_Value,"16,320.000"
Dist_AB_Road,745.322
Dist_Bus_Stop,396.288
Dist_Railway,332.220


,Label,Effect (%),Change (Rs.),End (Rs.)
0,"Reference property (654 sq ft, 1 floor, 25 yrs...",NaN,NaN,"15,999.714"
1,Number of floors = 3,-16.118,"-2,578.851","13,420.863"
2,Condition grade = Fair+,9.801,"1,315.366","14,736.229"
3,Total floor area (sq ft) = 684,4.473,659.186,"15,395.415"
4,Distance to A/B road (m) = 745,-1.627,-250.462,"15,144.953"
5,Distance to railway station (m) = 332,0.298,45.193,"15,190.146"
6,"Distance to hospital (m) = 1,418",0.288,43.762,"15,233.909"
7,Building age (years) = 25,0.000,0.000,"15,233.909"
8,Model estimate for this property,NaN,NaN,"15,233.909"


## 5. Supplementary checks

### 5.1 Spatially grouped cross-validation
The dataset has no road field, so a road-grouped validation (methodology 3.8) is approximated. The training
properties are grouped into 10 spatial clusters (k-means on their standardised GIS distances), and whole clusters are
held out. This tests whether the models carry over to **parts of the ward they were not trained on**, which random
CV does not test.

In [15]:
groups = M.spatial_groups(train)
sp = []
for s in ["A", "B"]:
    for k in C.ALGORITHMS:
        r = OUT[s]["results"][k]
        f = M.spatial_cv(k, train, ytr, r["features"], categories, r["params"], groups)
        sp.append({"Model set": C.MODEL_SETS[s], "Algorithm": C.MODEL_NAMES[k], "Spatial CV RMSE": f["RMSE"].mean(),
                   "Spatial CV MAE": f["MAE"].mean(), "Spatial CV R2": f["R2"].mean(),
                   "Random CV RMSE": r["cv"]["CV RMSE"]})
spatial = pd.DataFrame(sp)
spatial.to_csv(T / "AB_spatial_cv.csv", index=False)
print("Cluster sizes:", np.bincount(groups))
spatial

Cluster sizes: [77 69 57 16  8 53 31 38 13 39]


,Model set,Algorithm,Spatial CV RMSE,Spatial CV MAE,Spatial CV R2,Random CV RMSE
0,Model A - property only,Multiple Linear Regression,"13,370.861","4,357.475",0.558,"12,417.444"
1,Model A - property only,Random Forest,"12,643.353","4,691.385",0.747,"11,609.659"
2,Model A - property only,XGBoost,"12,611.574","4,524.425",0.766,"11,663.288"
3,Model B - property + GIS,Multiple Linear Regression,"14,503.511","4,544.461",0.423,"13,119.307"
4,Model B - property + GIS,Random Forest,"12,851.938","5,089.715",0.747,"11,468.491"
5,Model B - property + GIS,XGBoost,"13,469.732","5,473.385",0.734,"11,247.784"


### 5.2 Raw-target sensitivity (models fitted on Annual Value in rupees instead of log)

In [16]:
raw = []
for s in ["A", "B"]:
    for k in C.ALGORITHMS:
        r = OUT[s]["results"][k]
        m = M.build_model(k, r["features"], categories, r["params"], log_target=False).fit(train[r["features"]], ytr)
        p = m.predict(test[r["features"]])
        st = M.ratio_statistics(yte, p)
        raw.append({"Model set": C.MODEL_SETS[s], "Algorithm": C.MODEL_NAMES[k], **{f"Test {a}": b for a, b in M.metrics(yte, p).items()},
                    "Log-target test RMSE": r["test"]["RMSE"], "COD (%)": st["COD (%)"], "PRD": st["PRD"],
                    "Negative estimates": st["Negative estimates"]})
rawt = pd.DataFrame(raw)
rawt.to_csv(T / "AB_raw_target_sensitivity.csv", index=False)
rawt

,Model set,Algorithm,Test MAE,Test RMSE,Test R2,Log-target test RMSE,COD (%),PRD,Negative estimates
0,Model A - property only,Multiple Linear Regression,"8,025.340","10,209.491",0.818,"6,626.339",66.941,0.897,18
1,Model A - property only,Random Forest,"3,572.062","7,302.543",0.907,"6,905.540",25.077,1.162,0
2,Model A - property only,XGBoost,"3,277.738","6,595.746",0.924,"6,171.024",22.516,1.166,0
3,Model B - property + GIS,Multiple Linear Regression,"7,840.394","10,009.530",0.825,"6,345.987",62.971,0.910,18
4,Model B - property + GIS,Random Forest,"3,010.869","6,248.626",0.932,"5,973.176",22.269,1.147,0
5,Model B - property + GIS,XGBoost,"3,021.371","5,814.493",0.941,"5,767.699",21.590,1.132,0


The tuned hyperparameters from the log-target search are reused here, so this compares only the target scale.
It shows whether the main conclusions depend on the log transform.

## 6. Build the Chapter 4 draft

In [17]:
from src import report
path = report.build()
print("Chapter 4 draft written to", path)

Chapter 4 draft written to /Users/avishka/Documents/ML-model/final-model/report/Chapter4_draft.docx
